In [131]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
import json
import os
import re
from pathlib import Path
import time


text_path = Path('./output/tolstoy_cleaned.txt')
with open(text_path, 'r', encoding='utf-8') as f:
    full_text = f.read()
full_text = full_text
print(f"Загружено символов: {len(full_text):,}")
print(f"Первые 200 символов:\n{full_text[:200]}")

Загружено символов: 5,273,107
Первые 200 символов:
анна каренина мне отмщение, и аз воздам часть первая все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива посвоему. все смешалось в доме облонских. жена узнала, что муж был


In [132]:
def create_causal_mask(seq_len, device='cpu'):
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    mask = ~mask
    return mask

mask = create_causal_mask(5)
print(mask)

tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])


In [133]:
class PositionalEncoding(nn.Module):
    """Позиционное кодирование sin/cos"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.d_model = d_model
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * 
                           -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: [batch, seq_len, d_model]
        return x + self.pe[:x.size(1), :]


class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention"""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.d_model = d_model
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Линейные проекции и разделение на heads
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # scores shape: [batch, num_heads, seq_len, seq_len]
        
        # Применяем маску
        if mask is not None:
            # mask должен быть [seq_len, seq_len] или [1, 1, seq_len, seq_len]
            # Расширяем до [batch, num_heads, seq_len, seq_len]
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            # Убеждаемся, что mask имеет правильную форму для broadcasting
            # scores: [batch, num_heads, seq_len, seq_len]
            # mask: [1, 1, seq_len, seq_len]
            # Применяем маску: там где mask == 0, ставим -1e9
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        # Объединяем heads
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention с residual
        attn_out = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward с residual
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x


class TolstoyGPT(nn.Module):
    """Decoder-only Transformer для генерации текста"""
    def __init__(self, vocab_size, d_model=256, num_heads=8, 
                 num_layers=6, d_ff=1024, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        
        # Инициализация весов
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, x, mask=None):
        # x: [batch, seq_len]
        # Token embedding + positional encoding
        x = self.token_embedding(x)  # [batch, seq_len, d_model]
        x = self.pos_encoding(x)     # [batch, seq_len, d_model]
        
        # Проход через блоки
        for layer in self.layers:
            x = layer(x, mask)
        
        # Финальная нормализация и head
        x = self.norm(x)
        return self.head(x)  # [batch, seq_len, vocab_size]
    
    def generate(self, prompt, max_new_tokens=200, temperature=0.8, top_k=40):
        """Генерация текста"""
        self.eval()
        
        # Кодируем промпт
        context = torch.tensor([stoi.get(ch, 0) for ch in prompt], 
                               dtype=torch.long, device=device).unsqueeze(0)
        
        generated = list(prompt)
        
        with torch.no_grad():
            for _ in range(max_new_tokens):
                # Обрезаем до максимальной длины
                ctx = context[:, -SEQ_LEN:]
                seq_len = ctx.size(1)
                
                # Causal mask
                mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
                mask = ~mask  # True там, где можно смотреть
                mask = mask.to(device)
                
                # Получаем логиты
                logits = self(ctx, mask)  # [1, seq_len, vocab_size]
                logits = logits[0, -1, :] / temperature  # [vocab_size]
                
                # Top-k фильтрация
                if top_k is not None and top_k < logits.size(-1):
                    top_probs, top_indices = torch.topk(logits, top_k)
                    logits = torch.full_like(logits, float('-inf'))
                    logits[top_indices] = top_probs
                
                # Сэмплируем
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, 1).item()
                
                # Добавляем символ
                char = itos[next_token]
                generated.append(char)
                
                # Обновляем контекст
                context = torch.cat([context, torch.tensor([[next_token]], device=device)], dim=1)
        
        return ''.join(generated)

In [134]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Используется устройство: cuda
GPU: NVIDIA GeForce RTX 4060


In [ ]:
chars = sorted(list(set(full_text)))
vocab_size_char = len(chars)
print(f"Размер символьного словаря: {vocab_size_char}")

stoi_char = {ch: i for i, ch in enumerate(chars)}
itos_char = {i: ch for i, ch in enumerate(chars)}

def encode_char(s):
    return [stoi_char.get(ch, 0) for ch in s]

def decode_char(indices):
    return ''.join([itos_char[i] for i in indices])

class TolstoyDatasetChar(Dataset):
    def __init__(self, text, seq_len):
        self.seq_len = seq_len
        self.data = torch.tensor(encode_char(text), dtype=torch.long)
        self.n_samples = len(self.data) - seq_len
        print(f"Всего токенов (символов): {len(self.data):,}")
        print(f"Последовательностей: {self.n_samples:,}")
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        x = self.data[idx:idx + self.seq_len]
        y = self.data[idx + 1:idx + self.seq_len + 1]
        return x, y

SEQ_LEN_CHAR = 128
BATCH_SIZE_CHAR = 64

dataset_char = TolstoyDatasetChar(full_text, SEQ_LEN_CHAR)

train_size = int(0.9 * len(dataset_char))
val_size = len(dataset_char) - train_size
train_dataset_char, val_dataset_char = torch.utils.data.random_split(
    dataset_char, [train_size, val_size]
)

train_loader_char = DataLoader(train_dataset_char, batch_size=BATCH_SIZE_CHAR,
                               shuffle=True, drop_last=True)
val_loader_char = DataLoader(val_dataset_char, batch_size=BATCH_SIZE_CHAR,
                             shuffle=False, drop_last=True)



D_MODEL = 128
NUM_HEADS = 8
NUM_LAYERS = 6
D_FF = 512
DROPOUT = 0.1
LEARNING_RATE = 3e-4
EPOCHS = 1
model_char = TolstoyGPT(
    vocab_size=vocab_size_char,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT
).to(device)

print(f"Параметров: {sum(p.numel() for p in model_char.parameters()):,}")

Размер символьного словаря: 36
Всего токенов (символов): 5,273,107
Последовательностей: 5,272,979
Параметров: 1,199,140


In [136]:
optimizer = torch.optim.AdamW(model_char.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    # Train
    model_char.train()
    total_train_loss = 0
    start_time = time.time()
    
    for batch_idx, (x, y) in enumerate(train_loader_char):
        x, y = x.to(device), y.to(device)
        mask = create_causal_mask(SEQ_LEN_CHAR).to(device)
        
        logits = model_char(x, mask)
        loss = criterion(logits.view(-1, vocab_size_char), y.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_char.parameters(), 1.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if (batch_idx + 1) % 1000 == 0:
            print(f"  Эпоха {epoch+1}, Батч {batch_idx+1}/{len(train_loader_char)}, Loss: {loss.item():.4f}")
    
    avg_train_loss = total_train_loss / len(train_loader_char)
    train_losses.append(avg_train_loss)

    model_char.eval()
    total_val_loss = 0
    with torch.no_grad():
        for x, y in val_loader_char:
            x, y = x.to(device), y.to(device)
            mask = create_causal_mask(SEQ_LEN_CHAR).to(device)
            logits = model_char(x, mask)
            loss = criterion(logits.view(-1, vocab_size_char), y.view(-1))
            total_val_loss += loss.item()
    
    avg_val_loss = total_val_loss / len(val_loader_char)
    val_losses.append(avg_val_loss)
    
    elapsed = time.time() - start_time

    def generate_char(prompt, max_new=100, temp=0.8):
        model_char.eval()
        context = torch.tensor(encode_char(prompt), dtype=torch.long, device=device).unsqueeze(0)
        generated = list(prompt)
        
        with torch.no_grad():
            for _ in range(max_new):
                ctx = context[:, -SEQ_LEN_CHAR:]
                seq_len = ctx.size(1)
                mask = ~torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
                logits = model_char(ctx, mask)[0, -1, :] / temp
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, 1).item()
                char = itos_char[next_token]
                generated.append(char)
                context = torch.cat([context, torch.tensor([[next_token]], device=device)], dim=1)
        return ''.join(generated)
    
    sample = generate_char("Однажды", max_new=80, temp=0.7)
    print(f"\n📖 Пример:\n{sample}\n")
    print("-"*50)

print("✅ Обучение завершено!")


Начинаем обучение символьной модели...
  Эпоха 1, Батч 1000/74151, Loss: 2.2955
  Эпоха 1, Батч 2000/74151, Loss: 1.9977
  Эпоха 1, Батч 3000/74151, Loss: 1.8256
  Эпоха 1, Батч 4000/74151, Loss: 1.7615
  Эпоха 1, Батч 5000/74151, Loss: 1.6282
  Эпоха 1, Батч 6000/74151, Loss: 1.6203
  Эпоха 1, Батч 7000/74151, Loss: 1.5475
  Эпоха 1, Батч 8000/74151, Loss: 1.5023
  Эпоха 1, Батч 9000/74151, Loss: 1.5335
  Эпоха 1, Батч 10000/74151, Loss: 1.5341
  Эпоха 1, Батч 11000/74151, Loss: 1.4491
  Эпоха 1, Батч 12000/74151, Loss: 1.4325
  Эпоха 1, Батч 13000/74151, Loss: 1.3959
  Эпоха 1, Батч 14000/74151, Loss: 1.4543
  Эпоха 1, Батч 15000/74151, Loss: 1.4358
  Эпоха 1, Батч 16000/74151, Loss: 1.4190
  Эпоха 1, Батч 17000/74151, Loss: 1.3678
  Эпоха 1, Батч 18000/74151, Loss: 1.3864
  Эпоха 1, Батч 19000/74151, Loss: 1.3892
  Эпоха 1, Батч 20000/74151, Loss: 1.4216
  Эпоха 1, Батч 21000/74151, Loss: 1.3831
  Эпоха 1, Батч 22000/74151, Loss: 1.3634
  Эпоха 1, Батч 23000/74151, Loss: 1.3678
  Э

In [137]:
import os
import json
from pathlib import Path

os.makedirs('./checkpoints', exist_ok=True)
os.makedirs('./output', exist_ok=True)

torch.save(model_char.state_dict(), './checkpoints/tolstoy_final.pt')

with open('./output/vocab.json', 'w', encoding='utf-8') as f:
    json.dump({
        'stoi': stoi_char,
        'itos': {str(k): v for k, v in itos_char.items()}
    }, f, ensure_ascii=False, indent=2)

config = {
    'vocab_size': vocab_size_char,
    'd_model': D_MODEL,
    'num_heads': NUM_HEADS,
    'num_layers': NUM_LAYERS,
    'd_ff': D_FF,
    'dropout': DROPOUT,
    'seq_len': SEQ_LEN_CHAR,
    'epochs': EPOCHS
}

with open('./checkpoints/model_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

checkpoint = {
    'epoch': EPOCHS,
    'model_state_dict': model_char.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': avg_train_loss,
    'val_loss': avg_val_loss,
    'config': config
}

torch.save(checkpoint, './checkpoints/tolstoy_checkpoint.pt')